## Environment (TabPHM `.venv`)

`python.defaultInterpreterPath` in [`.vscode/settings.json`](../../.vscode/settings.json) does **not** force the **notebook kernel**. If you see **`ModuleNotFoundError: awkward`**, the notebook is still using another interpreter (e.g. conda **`tb`** — check the next cell’s `sys.executable`).

**Fix the kernel (pick one):**

1. **Command Palette** → `Notebook: Select Notebook Kernel` (or click the kernel name in the notebook toolbar).
2. Prefer **`TabPHM (.venv)`** under **Jupyter Kernels** — that is the `tabphm` kernelspec registered inside `.venv`.
3. Or under **Python Environments**, choose **`./.venv/bin/python`** for this workspace — **not** `anaconda3/envs/tb`.

If **`TabPHM (.venv)`** is missing, run once from the repo root:

`./scripts/register_tabphm_jupyter_kernel.sh`

Then **Developer: Reload Window** and select the kernel again.


In [1]:
import sys
from pathlib import Path

# Do not call .resolve(): many .venv/bin/python executables symlink to a base
# interpreter; resolving would hide ".venv" in the path and break this check.
exe = Path(sys.executable)
using_project_venv = exe.parent.name == "bin" and exe.parent.parent.name == ".venv"
print("sys.executable:", exe)
if not using_project_venv:
    print(
        "\n>>> Wrong kernel (not project .venv). Example: conda env `tb`.\n"
        "    Notebook: Select Notebook Kernel → `TabPHM (.venv)` or `./.venv/bin/python`.\n"
    )
else:
    print("OK: interpreter is a .venv/bin/python (expected project env).")

import awkward as ak  # noqa: F401 — fails if the kernel is not the TabPHM .venv

print("OK: awkward import succeeded.")


sys.executable: /Users/leone/projects/TabPHM/.venv/bin/python
OK: interpreter is a .venv/bin/python (expected project env).
OK: awkward import succeeded.


In [ ]:
# XJTU-SYLoader: predefined splits from PHMD
#
# - PHMD exposes each task as a list of **folds**; `fold` picks one fold index.
# - That fold is already split into **train / val / test** DataFrames (by unit rows).
# - `PHMDMultiSourceLoader` groups by `bearing`, runs `_process_unit` per unit, and
#   builds a `SplitDatasetContainer`: keys like `features`, `target`; each maps to
#   `{train, val, test}` with a **list of one payload per unit** (multi-unit).
# - `payload_cache_path` stores the processed loader payload; the next run skips PHMD
#   CSV read + `_process_unit` when `get_cache_fingerprint()` still matches.

import os
from pathlib import Path

from picid.data.datasources.phmd_xjtu_sy import XJTU_SYLoader

# Extracted dataset tree (CSV bearings). Default: your PHMD extract path.
XJTU_EXTRACT_DIR = Path(
    os.environ.get(
        "XJTU_SY_EXTRACT_DIR",
        "/Users/leone/projects/TabPHM/datasets/phmd_cache/datasets/XJTU-SY",
    )
)
# PHMD expects cache_dir such that raw files live at cache_dir/datasets/XJTU-SY/
cache_dir = XJTU_EXTRACT_DIR.parent.parent
if not (cache_dir / "datasets" / "XJTU-SY").is_dir():
    raise FileNotFoundError(
        f"Expected PHMD layout {cache_dir}/datasets/XJTU-SY/ — check XJTU_SY_EXTRACT_DIR ({XJTU_EXTRACT_DIR})"
    )

PICID_PROCESSED = Path(
    os.environ.get(
        "XJTU_SY_PICID_CACHE",
        XJTU_EXTRACT_DIR / "_picid_xjtu_fold0_rul_ragged.joblib",
    )
)

print("PHMD cache_dir (contains datasets/XJTU-SY):", cache_dir.resolve())
print("PicID processed payload cache:", PICID_PROCESSED.resolve())

xjtu = XJTU_SYLoader(
    fold=0,
    data_name="XJTU-SY",
    task_mode="rul",
    auxiliary_tasks=[],
    cache_dir=str(cache_dir),
    use_ragged=True,
    payload_cache_path=str(PICID_PROCESSED),
)
xjtu.load_data()
# split_data() is a no-op: splits are predefined by PHMD.
data = xjtu.get_data()
meta = xjtu.get_meta_data()
print("meta keys:", sorted(meta.keys()))

split_keys = list(data.features.keys())
print("Split keys:", split_keys)
for split in split_keys:
    units = data.features[split]
    print(f"  {split}: n_units={len(units)}")
    if units:
        u0 = units[0]
        print(f"    first unit features type={type(u0).__name__} len={len(u0)}")



PHMD cache_dir (contains datasets/XJTU-SY): /Users/leone/projects/TabPHM/datasets/phmd_cache
PicID processed cache: /Users/leone/projects/TabPHM/datasets/phmd_cache/datasets/XJTU-SY/_picid_xjtu_fold0_rul_ragged.joblib
Dataset XJTU-SY already downloaded and extracted
Remember to cite the original publisher dataset:
	@article{wang2018hybrid,
	  title={A hybrid prognostics approach for estimating remaining useful life of rolling element bearings},
	  author={Wang, Biao and Lei, Yaguo and Li, Naipeng and Li, Ningbo},
	  journal={IEEE Transactions on Reliability},
	  volume={69},
	  number={1},
	  pages={401--412},
	  year={2018},
	  publisher={IEEE}
	}
You can download the dataset manually from:  https://biaowang.tech/xjtu-sy-bearing-datasets/

** If you find this tool useful, please cite our SoftwareX paper: 
	Solís-Martín, David, Juan Galán-Páez, and Joaquín Borrego-Díaz. "PHMD: An easy data access tool for prognosis and health management datasets." SoftwareX 29 (2025): 102039.



Reading Bearing3_1:  39%|███▉      | 3630/9216 [00:43<00:51, 108.01it/s]Reading Bearing3_1:  50%|█████     | 4625/9216 [00:55<01:01, 75.03it/s] Reading Bearing3_5: 100%|██████████| 9216/9216 [02:25<00:00, 63.24it/s] 


In [ ]:
# XJTU-SY: unit_id and unit_name within each split
# Populated from each unit's `metadata` in the loader (`SplitDatasetContainer.unit_metadata`).
split_keys = list(data.features.keys())
for split in split_keys:
    units_meta = data.unit_metadata[split]
    print(f"{split} ({len(units_meta)} units):")
    for i, m in enumerate(units_meta):
        uid = m.get("unit_id")
        uname = m.get("unit_name")
        print(f"  [{i}] unit_id={uid!r}  unit_name={uname!r}")


In [1]:
# PRONOSTIA: same split / unit listing (see configs/datasource/pronostia.yaml)

import os
from pathlib import Path
from picid.data.datasources.phmd_pronostia import PronostiaLoader
REPO = Path("/Users/leone/projects/TabPHM")  # change if your clone lives elsewhere
cache_dir = Path(os.environ.get("PHMD_CACHE_DIR", REPO / "datasets" / "phmd_cache"))
cache_dir.mkdir(parents=True, exist_ok=True)
print("PHMD cache_dir:", cache_dir.resolve())
pron = PronostiaLoader(
    fold=0,
    data_name="PRONOSTIA",
    task_mode="rul",
    cache_dir=str(cache_dir),
    use_ragged=True,
)
pron.load_data()
pdata = pron.get_data()
print("PRONOSTIA loader meta keys:", sorted(pron.get_meta_data().keys()))

p_split_keys = list(pdata.features.keys())
print("Split keys:", p_split_keys)
for split in p_split_keys:
    n_feat = len(pdata.features[split])
    units_meta = pdata.unit_metadata[split]
    print(f"{split} ({n_feat} units):")
    for i, m in enumerate(units_meta):
        print(f"  [{i}] unit_id={m.get('unit_id')!r}  unit_name={m.get('unit_name')!r}")


PHMD cache_dir: /Users/leone/projects/TabPHM/datasets/phmd_cache
Dataset PRONOSTIA already downloaded and extracted
Dataset PRONOSTIA already downloaded and extracted
Remember to cite the original publisher dataset:
	@inproceedings{nectoux2012pronostia,    
	    title={PRONOSTIA: An experimental platform for bearings accelerated degradation tests.},    
	    author={Nectoux, Patrick and Gouriveau, Rafael and Medjaher, Kamal and Ramasso, Emmanuel and Chebel-Morello, Brigitte and Zerhouni, Noureddine and Varnier, Christophe},    
	    booktitle={IEEE International Conference on Prognostics and Health Management, PHM'12.},    
	    pages={1--8},    
	    year={2012},    
	    organization={IEEE Catalog Number: CPF12PHM-CDR}    
	}
You can download the dataset manually from:  https://www.nasa.gov/intelligent-systems-division/discovery-and-systems-health/pcoe/pcoe-data-set-repository/

** If you find this tool useful, please cite our SoftwareX paper: 
	Solís-Martín, David, Juan Galán-Páez, 

Reading Bearing3_2: 100%|█████████▉| 7528/7534 [00:18<00:00, 416.23it/s]


Number of files loaded per bearing, each file consists 10 sec.
Key                  | Value     
---------------------------------
Bearing1_1 | 2802.00   
Bearing1_2 | 870.00    
Bearing2_1 | 910.00    
Bearing2_2 | 796.00    
Bearing3_1 | 514.00    
Bearing3_2 | 1636.00   


Reading Bearing3_3: 100%|█████████▉| 13938/13959 [00:31<00:00, 444.60it/s]


Number of files loaded per bearing, each file consists 10 sec.
Key                  | Value     
---------------------------------
Bearing1_3 | 1801.00   
Bearing1_4 | 1128.00   
Bearing1_5 | 2301.00   
Bearing1_6 | 2301.00   
Bearing1_7 | 1501.00   
Bearing2_3 | 1201.00   
Bearing2_4 | 611.00    
Bearing2_5 | 2001.00   
Bearing2_6 | 571.00    
Bearing2_7 | 171.00    
Bearing3_3 | 351.00    


INFO:root:Read in 53.05058670043945 seconds
INFO:root:Read 3 sets: train,val,test
INFO:root:Columns: H_acc,unit,V_acc,rul
INFO:root:Train shape: (10060800, 4)
INFO:root:Val shape: (9210880, 4)
INFO:root:Test shape: (35681280, 4)
INFO:picid.data.datasources.base.phmd_loader:Number of train units: 4
INFO:picid.data.datasources.base.phmd_loader:Number of val units: 2
INFO:picid.data.datasources.base.phmd_loader:Number of test units: 11


PRONOSTIA loader meta keys: ['features', 'identifier', 'min_ts_len', 'nature', 'num_units', 'target', 'type', 'unit_ids', 'unit_names']
Split keys: ['train', 'val', 'test']
train (4 units):
  [0] unit_id=(1, 2)  unit_name='Bearing 1_2'
  [1] unit_id=(2, 1)  unit_name='Bearing 2_1'
  [2] unit_id=(3, 1)  unit_name='Bearing 3_1'
  [3] unit_id=(3, 2)  unit_name='Bearing 3_2'
val (2 units):
  [0] unit_id=(1, 1)  unit_name='Bearing 1_1'
  [1] unit_id=(2, 2)  unit_name='Bearing 2_2'
test (11 units):
  [0] unit_id=(1, 3)  unit_name='Bearing 1_3'
  [1] unit_id=(1, 4)  unit_name='Bearing 1_4'
  [2] unit_id=(1, 5)  unit_name='Bearing 1_5'
  [3] unit_id=(1, 6)  unit_name='Bearing 1_6'
  [4] unit_id=(1, 7)  unit_name='Bearing 1_7'
  [5] unit_id=(2, 3)  unit_name='Bearing 2_3'
  [6] unit_id=(2, 4)  unit_name='Bearing 2_4'
  [7] unit_id=(2, 5)  unit_name='Bearing 2_5'
  [8] unit_id=(2, 6)  unit_name='Bearing 2_6'
  [9] unit_id=(2, 7)  unit_name='Bearing 2_7'
  [10] unit_id=(3, 3)  unit_name='Bearing 

In [5]:
# PRONOSTIA — iterate all units and group by Table 10 (Learning vs Test)
# PHMD fold 0 still places “Learning” bearings into loader train/val; this cell
# re-labels every unit by the paper split and prints tensor shapes.
from collections import defaultdict

import awkward as ak
import numpy as np

TABLE10_LEARNING = frozenset(
    f"Bearing {c}" for c in ("1_1", "1_2", "2_1", "2_2", "3_1", "3_2")
)
TABLE10_TEST = frozenset(
    f"Bearing {c}"
    for c in (
        "1_3",
        "1_4",
        "1_5",
        "1_6",
        "1_7",
        "2_3",
        "2_4",
        "2_5",
        "2_6",
        "2_7",
        "3_3",
    )
)
# Table 10 — runtime (s) and actual RUL at experiment end (s); learning RUL = 0
TABLE10_RUNTIME_S = {
    "Bearing 1_1": 28020,
    "Bearing 1_2": 8700,
    "Bearing 2_1": 9100,
    "Bearing 2_2": 7960,
    "Bearing 3_1": 5140,
    "Bearing 3_2": 16360,
    "Bearing 1_3": 18010,
    "Bearing 1_4": 11280,
    "Bearing 1_5": 23010,
    "Bearing 1_6": 23010,
    "Bearing 1_7": 15010,
    "Bearing 2_3": 12010,
    "Bearing 2_4": 6110,
    "Bearing 2_5": 20010,
    "Bearing 2_6": 5710,
    "Bearing 2_7": 1710,
    "Bearing 3_3": 3510,
}
TABLE10_RUL_S = {
    "Bearing 1_3": 5730,
    "Bearing 1_4": 2900,
    "Bearing 1_5": 1610,
    "Bearing 1_6": 1460,
    "Bearing 1_7": 7570,
    "Bearing 2_3": 7530,
    "Bearing 2_4": 1390,
    "Bearing 2_5": 3090,
    "Bearing 2_6": 1290,
    "Bearing 2_7": 580,
    "Bearing 3_3": 820,
}


def _first_seg_shape(x):
    if not isinstance(x, ak.Array) or len(x) == 0:
        return None
    try:
        return tuple(ak.to_numpy(x[0]).shape)
    except Exception:
        return str(x[0].type)


rows = []
by_table10 = defaultdict(list)
loader_splits = list(pdata.features.keys())

for split in loader_splits:
    n = len(pdata.unit_metadata[split])
    for unit_idx in range(n):
        m = pdata.unit_metadata[split][unit_idx]
        name = m["unit_name"]
        if name in TABLE10_LEARNING:
            t10 = "Learning"
        elif name in TABLE10_TEST:
            t10 = "Test"
        else:
            t10 = "Unknown"

        feats = pdata.features[split][unit_idx]
        targ = pdata.target[split][unit_idx]
        n_seq = len(feats) if isinstance(feats, ak.Array) else None
        seg_shape = _first_seg_shape(feats)

        rt = TABLE10_RUNTIME_S.get(name)
        rul = TABLE10_RUL_S.get(name) if t10 == "Test" else None

        row = {
            "table10": t10,
            "unit_name": name,
            "loader_split": split,
            "unit_idx": unit_idx,
            "n_sequences": n_seq,
            "segment_shape": seg_shape,
            "table_runtime_s": rt,
            "table_rul_s": rul,
            "loader_unit_length": m.get("unit_length"),
        }
        rows.append(row)
        by_table10[t10].append(row)

# Summary: Table 10 groups (order: learning bearings, then test)
print("=== By Table 10 (paper) ===")
for group in ("Learning", "Test", "Unknown"):
    items = sorted(by_table10[group], key=lambda r: r["unit_name"])
    print(f"\n{group} ({len(items)} units in loader payload):")
    for r in items:
        extra = ""
        if r["table_runtime_s"] is not None:
            extra += f" | Table10 runtime={r['table_runtime_s']}s"
        if r["table10"] == "Test" and r["table_rul_s"] is not None:
            extra += f" | Table10 RUL={r['table_rul_s']}s"
        print(
            f"  {r['unit_name']}: PHMD loader_split={r['loader_split']!r}, idx={r['unit_idx']}, "
            f"n_seq={r['n_sequences']}, seg0={r['segment_shape']}{extra}"
        )

# Full table (all points)
print("\n=== All units (flat iteration) ===")
for r in sorted(rows, key=lambda x: (x["table10"], x["unit_name"])):
    print(
        f"{r['table10']:<9}  {r['unit_name']:<14}  loader={r['loader_split']:<5}  "
        f"n_seq={r['n_sequences']}  seg0={r['segment_shape']}"
    )


=== By Table 10 (paper) ===

Learning (6 units in loader payload):
  Bearing 1_1: PHMD loader_split='val', idx=0, n_seq=2802, seg0=(2560, 2) | Table10 runtime=28020s
  Bearing 1_2: PHMD loader_split='train', idx=0, n_seq=870, seg0=(2560, 2) | Table10 runtime=8700s
  Bearing 2_1: PHMD loader_split='train', idx=1, n_seq=910, seg0=(2560, 2) | Table10 runtime=9100s
  Bearing 2_2: PHMD loader_split='val', idx=1, n_seq=796, seg0=(2560, 2) | Table10 runtime=7960s
  Bearing 3_1: PHMD loader_split='train', idx=2, n_seq=514, seg0=(2560, 2) | Table10 runtime=5140s
  Bearing 3_2: PHMD loader_split='train', idx=3, n_seq=1636, seg0=(2560, 2) | Table10 runtime=16360s

Test (11 units in loader payload):
  Bearing 1_3: PHMD loader_split='test', idx=0, n_seq=1801, seg0=(2560, 2) | Table10 runtime=18010s | Table10 RUL=5730s
  Bearing 1_4: PHMD loader_split='test', idx=1, n_seq=1128, seg0=(2560, 2) | Table10 runtime=11280s | Table10 RUL=2900s
  Bearing 1_5: PHMD loader_split='test', idx=2, n_seq=2301, seg

In [3]:
# Same idea for XJTU-SY (first training unit) — segment size is 32768 in the loader
split_x = "train"
ux = 0
mx = data.unit_metadata[split_x][ux]
fx = data.features[split_x][ux]
tx = data.target[split_x][ux]
print(f"--- {mx.get('unit_name')} | split={split_x!r} | unit_length={mx.get('unit_length')} ---")
if isinstance(fx, ak.Array) and len(fx) > 0:
    try:
        print(f"features[0] shape: {ak.to_numpy(fx[0]).shape}")
    except Exception:
        print(f"features[0] type: {fx[0].type}")
if isinstance(tx, ak.Array) and len(tx) > 0:
    try:
        print(f"target[0] shape: {ak.to_numpy(tx[0]).shape}")
    except Exception:
        print(f"target[0] type: {tx[0].type}")


NameError: name 'data' is not defined

<Array [[0.056, 0.769], [...], ..., [-0.022, 0.533]] type='2560 * 2 * float32'>